# 04 - Training & evaluation across all split schemes - CIC-IDS-2017

In [1]:
DATASET = "cic2017"
FT_ROOT = None
TARGET_PER_CLASS = 50_000
MODELS_TO_RUN = None
RUN_CV = True

In [2]:
import sys
from pathlib import Path

def _find_nb_common():
    for cand in [Path.cwd(), *Path.cwd().parents]:
        for c in (cand, cand / "notebooks"):
            if (c / "nb_common.py").exists():
                return c
    raise FileNotFoundError("nb_common.py not found relative to " + str(Path.cwd()))
sys.path.insert(0, str(_find_nb_common()))

import time
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

import nb_common as nbc
from ids import config
from ids.schema import CANONICAL_COLUMNS
from ids.models.pipelines import build_pipeline
from ids.models.registry import MODEL_NAMES
from ids.evaluation.metrics import evaluate_model
from ids.evaluation.cross_val import cross_validate_models, summarize_cv
from ids.evaluation.plots import plot_confusion_matrix

P = nbc.ft_paths(DATASET, FT_ROOT)
log = nbc.setup_logging(P.logs / "04_experiments.log", "nb04")

df = pd.read_parquet(P.data / "clean.parquet")
X, y = df[CANONICAL_COLUMNS], df["y"].to_numpy()
top40 = nbc.load_json(P.features / "top40.json")["selected"]
mapping = nbc.load_json(P.data / "label_mapping.json")
class_names = [c for c, _ in sorted(mapping.items(), key=lambda kv: kv[1])]
n_classes = len(class_names)
models_to_run = MODELS_TO_RUN or list(MODEL_NAMES)
log.info("%d rows | %d selected features | %d classes | models=%s | target_per_class=%d",
         len(X), len(top40), n_classes, models_to_run, TARGET_PER_CLASS)

2026-07-06 10:38:08,332 INFO nb04: 2520798 rows | 40 selected features | 15 classes | models=['decision_tree', 'random_forest', 'svm', 'naive_bayes', 'knn'] | target_per_class=50000


## 1. Build all split-scheme indices

In [3]:
idx = np.arange(len(X))

def holdout(test_size):
    tr, te = train_test_split(idx, test_size=test_size, stratify=y,
                              random_state=config.RANDOM_STATE)
    return {"train_idx": tr, "test_idx": te}

def train_val_test():
    tr_val, te = train_test_split(idx, test_size=0.10, stratify=y,
                                  random_state=config.RANDOM_STATE)
    tr, val = train_test_split(tr_val, test_size=2/9, stratify=y[tr_val],
                               random_state=config.RANDOM_STATE)
    return {"train_idx": tr, "val_idx": val, "test_idx": te}

SPLITS = {}
p_7030 = P.splits / "70_30_indices.npz"
SPLITS["70_30"] = dict(np.load(p_7030)) if p_7030.exists() else holdout(0.30)
SPLITS["80_20"] = holdout(0.20)
SPLITS["90_10"] = holdout(0.10)
SPLITS["70_20_10"] = train_val_test()

for scheme, s in SPLITS.items():
    np.savez_compressed(P.splits / f"{scheme}_indices.npz", **s)
    log.info("%s: %s", scheme,
             " / ".join(f"{k.replace('_idx','')}={len(v)}" for k, v in s.items()))

2026-07-06 10:38:11,694 INFO nb04: 70_30: train=1764558 / test=756240
2026-07-06 10:38:13,028 INFO nb04: 80_20: train=2016638 / test=504160
2026-07-06 10:38:14,375 INFO nb04: 90_10: train=2268718 / test=252080
2026-07-06 10:38:15,802 INFO nb04: 70_20_10: train=1764558 / val=504160 / test=252080


## 2. Train and evaluate every scheme x model combination

In [4]:
def run_one(scheme, name, train_idx, test_idx, val_idx=None):
    out_dir = P.runs / scheme / name
    out_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = out_dir / "metrics.json"
    if metrics_path.exists():
        log.info("SKIP %s/%s — metrics.json exists", scheme, name)
        return nbc.load_json(metrics_path)

    pipeline = build_pipeline(name, top40, target_per_class=TARGET_PER_CLASS)
    t0 = time.perf_counter()
    pipeline.fit(X.iloc[train_idx], y[train_idx])
    fit_seconds = time.perf_counter() - t0
    log.info("%s/%s: fit on %d rows in %.1fs", scheme, name, len(train_idx), fit_seconds)

    record = {"dataset": DATASET, "scheme": scheme, "model": name,
              "n_train": int(len(train_idx)), "n_test": int(len(test_idx)),
              "target_per_class": TARGET_PER_CLASS, "fit_seconds": round(fit_seconds, 3)}
    evals = [("test", test_idx, "")]
    if val_idx is not None:
        record["n_val"] = int(len(val_idx))
        evals.append(("validation", val_idx, "_val"))

    for split_name, eidx, suffix in evals:
        m, _, y_pred, y_proba = evaluate_model(name, pipeline, X.iloc[eidx], y[eidx])
        cm = confusion_matrix(y[eidx], y_pred, labels=np.arange(n_classes))
        np.save(out_dir / f"confusion_matrix{suffix}.npy", cm)
        np.save(out_dir / f"predictions{suffix}.npy", y_pred)
        if y_proba is not None:
            np.save(out_dir / f"probas{suffix}.npy", y_proba.astype(np.float32))
        rep = classification_report(y[eidx], y_pred, labels=np.arange(n_classes),
                                    target_names=class_names, output_dict=True,
                                    zero_division=0)
        pd.DataFrame(rep).T.to_csv(out_dir / f"per_class{suffix}.csv")
        if split_name == "test":
            plot_confusion_matrix(f"{name} ({scheme})", cm, class_names, out_dir)
        record[split_name] = m.to_dict()
        log.info("%s/%s [%s]: acc=%.4f f1_macro=%.4f roc_auc=%s det=%.3fms",
                 scheme, name, split_name, m.accuracy, m.f1_macro,
                 "n/a" if m.roc_auc is None else round(m.roc_auc, 4),
                 m.detection_time_ms)

    nbc.save_json(record, metrics_path)
    return record

records = []
with nbc.timed(log, "all holdout schemes"):
    for scheme, s in SPLITS.items():
        with nbc.timed(log, f"scheme {scheme}"):
            for name in models_to_run:
                records.append(run_one(scheme, name, s["train_idx"], s["test_idx"],
                                       s.get("val_idx")))
pd.DataFrame([{ "scheme": r["scheme"], "model": r["model"],
                "fit_s": r["fit_seconds"], **{k: round(v, 4) if isinstance(v, float) else v
                for k, v in r["test"].items() if k not in ("name",)}} for r in records])

2026-07-06 10:38:15,812 INFO nb04: START all holdout schemes
2026-07-06 10:38:15,815 INFO nb04: START scheme 70_30
2026-07-06 10:38:15,816 INFO nb04: SKIP 70_30/decision_tree — metrics.json exists
2026-07-06 10:38:15,817 INFO nb04: SKIP 70_30/random_forest — metrics.json exists
2026-07-06 10:38:15,818 INFO nb04: SKIP 70_30/svm — metrics.json exists
2026-07-06 10:38:15,818 INFO nb04: SKIP 70_30/naive_bayes — metrics.json exists
2026-07-06 10:38:15,820 INFO nb04: SKIP 70_30/knn — metrics.json exists
2026-07-06 10:38:15,821 INFO nb04: END   scheme 70_30 (0.0s)
2026-07-06 10:38:15,821 INFO nb04: START scheme 80_20
2026-07-06 10:38:15,821 INFO nb04: SKIP 80_20/decision_tree — metrics.json exists
2026-07-06 10:38:15,822 INFO nb04: SKIP 80_20/random_forest — metrics.json exists
2026-07-06 10:38:15,823 INFO nb04: SKIP 80_20/svm — metrics.json exists
2026-07-06 10:38:15,823 INFO nb04: SKIP 80_20/naive_bayes — metrics.json exists
2026-07-06 10:38:15,824 INFO nb04: SKIP 80_20/knn — metrics.json e

,scheme,model,fit_s,accuracy,precision_macro,recall_macro,f1_macro,roc_auc,detection_time_ms,n_test
0,70_30,decision_tree,14.211,0.9949,0.7062,0.8744,0.7583,0.9976,0.0003,756240
1,70_30,random_forest,34.076,0.9958,0.7673,0.8852,0.8019,0.9999,0.0043,756240
2,70_30,svm,12.681,0.6921,0.2891,0.8236,0.3272,0.9709,0.0008,756240
3,70_30,naive_bayes,2.810,0.3299,0.3809,0.6942,0.3438,0.9556,0.0007,756240
4,70_30,knn,2.318,0.9716,0.5915,0.9098,0.6546,0.9925,0.4476,756240
5,80_20,decision_tree,15.132,0.9955,0.7273,0.9142,0.7828,0.9979,0.0002,504160
6,80_20,random_forest,34.272,0.9959,0.8315,0.8990,0.8352,0.9999,0.0026,504160
7,80_20,svm,9.772,0.6984,0.2922,0.7928,0.3294,0.9732,0.0006,504160
8,80_20,naive_bayes,3.316,0.3193,0.3786,0.6814,0.3317,0.9532,0.0008,504160
9,80_20,knn,2.818,0.9714,0.5925,0.9266,0.6559,0.9925,0.4682,504160


## 3. Five-fold stratified cross-validation (scheme `cv5`)

In [5]:
cv_dir = P.runs / "cv5"
cv_dir.mkdir(parents=True, exist_ok=True)
cv_scores_path = cv_dir / "cv_scores.csv"

if not RUN_CV:
    log.info("RUN_CV=False — skipping")
elif cv_scores_path.exists():
    log.info("SKIP cv5 — cv_scores.csv exists")
    cv_summary = pd.read_csv(cv_dir / "cv_summary.csv")
else:
    with nbc.timed(log, "5-fold stratified CV, all models"):
        scores, _ = cross_validate_models(X, y, top40, model_names=models_to_run,
                                          n_splits=5, target_per_class=TARGET_PER_CLASS)
    scores.to_csv(cv_scores_path, index=False)
    cv_summary = summarize_cv(scores)
    cv_summary.to_csv(cv_dir / "cv_summary.csv", index=False)
cv_summary if RUN_CV else None

2026-07-06 10:38:15,858 INFO nb04: SKIP cv5 — cv_scores.csv exists


,model,accuracy_mean,accuracy_std,precision_macro_mean,precision_macro_std,recall_macro_mean,recall_macro_std,f1_macro_mean,f1_macro_std,roc_auc_mean,roc_auc_std
0,decision_tree,0.994488,0.000289,0.716156,0.008034,0.915427,0.016978,0.770081,0.006914,0.997854,0.000085
1,knn,0.971310,0.000370,0.588348,0.016530,0.899891,0.014171,0.649471,0.013813,0.992516,0.000209
2,naive_bayes,0.337202,0.006170,0.357047,0.032887,0.690565,0.029576,0.329340,0.023656,0.951858,0.001908
3,random_forest,0.995585,0.000149,0.809403,0.012673,0.905624,0.010774,0.825238,0.006564,0.999922,0.000016
4,svm,0.686036,0.004715,0.285869,0.004228,0.785568,0.018835,0.317438,0.007639,0.972017,0.000988


In [6]:
n_files = sum(1 for _ in P.runs.rglob("*") if _.is_file())
log.info("notebook 04 complete — %d artifact files under %s", n_files, P.runs)

2026-07-06 10:38:15,876 INFO nb04: notebook 04 complete — 142 artifact files under /Users/MAC/Projects/IDS/backend/artifacts/ft40/cic2017/runs
